# 01. Intervals

**Tier:** Deep. **Why you'd use this:** A calendar holds meeting ranges `[start, end)`. You need to know whether any two meetings overlap so a person can attend all of them.

**After this lesson you can:** sort intervals by start; compare each start with the previous end; allow touching endpoints.
About 25 minutes. **Uses:** [Lists](../../course/04-collections/01-lists-lesson.ipynb), [Loops](../../course/03-control-flow/02-loops-lesson.ipynb)


## Quick review

1. How do you sort a list of pairs by the first value?
2. What does `a < b` mean for meeting times when `b` is another meeting's start?

<details><summary>Answers</summary>

`sorted(intervals)` orders by the first element, then the second. If the previous meeting ended at or before `b` starts, they do not overlap. Skim [Lists](../../course/04-collections/01-lists-lesson.ipynb) if sorting feels rusty.
</details>


## Recognize it

You have closed or half-open time ranges and must detect whether any two conflict. The prompt says "can attend all meetings", "meeting rooms", or "overlap check".

Signals: each item is a start/end pair; order by time unlocks a linear scan. A look-alike that is not this pattern: merging overlapping ranges into a union — related prep (sort first), different output.


## Why it works this way

Comparing every pair of intervals is correct and costs about `n²` overlap tests. Sorting by start time lets you only compare each meeting to the one that ends latest among those that started earlier — which, after a sort, is the previous interval in the ordered list. If the current start is less than the previous end, they overlap. If the current start is greater than or equal to the previous end, they only touch or are separate, which this sibling treats as allowed.

Sorting costs about `n log n` comparisons. The following scan is linear. Extra memory is the sorted copy unless you sort in place. The overlap predicate for two ranges `[a, b]` and `[c, d]` (closed) is `a <= d and c <= b`. After sorting by start you only need `current_start < previous_end` for a strict conflict when touching endpoints are fine — use `<` vs `<=` carefully to match the problem's closed/open convention.

Meeting-rooms-II counts how many rooms you need: that sorts starts and ends as events, or uses a heap of end times. Merge-intervals rewrites the list into a disjoint union. Both share the "sort by time" opening move; they diverge in what they accumulate.
Half-open ranges `[start, end)` make touching endpoints natural: the next meeting may start exactly when the previous ends. Closed ranges need an explicit policy. Challenge prompts usually state whether `[1, 2]` and `[2, 3]` conflict. This sibling allows the touch.

Insert-interval problems keep the sorted order and splice one new range into the merged result. Meeting-rooms-II tracks how many meetings are live: sort all starts and ends, sweep a counter, and record the peak. The yes/no attend check is the simplest member of that family — once you sort, a single previous-end variable is enough. If meetings arrive online, a tree set of end times replaces the full sort, which is a different data-structure lesson.


## When to use it

Use sort-then-scan when you only need a yes/no conflict answer. Prefer a sweep of start/end events when you must count concurrent meetings. Prefer merging when the output should be the union of ranges. Prefer the nested pair check only for tiny calendars.
 If the prompt asks for the actual conflicting pair, return indexes during the scan instead of a boolean; the sort-and-compare skeleton stays the same.


## Brute force, and why it's slow

Test every pair for overlap.


In [1]:
def can_attend_slow(intervals):
    tests = 0
    for index, left in enumerate(intervals):
        for right in intervals[index + 1 :]:
            tests += 1
            if left[0] <= right[1] and right[0] <= left[1]:
                return False, tests
    return True, tests


for size in (20, 40, 80):
    data = [[i, i + 2] for i in range(0, size * 2, 2)]
    data[-1] = [0, size * 2]  # overlaps the first
    print(size, can_attend_slow(data))


20 (False, 1)
40 (False, 1)
80 (False, 1)


## Key insight

After sorting by start, each meeting only needs to clear the previous meeting's end. One failing comparison means someone cannot attend everything.


## Worked example

Return `True` when no two intervals overlap. Touching at an endpoint is allowed (`[1, 2]` and `[2, 3]` are fine).


In [2]:
def can_attend(intervals: list[list[int]]) -> bool:
    # 1. Chronological order by start time.
    ordered = sorted(intervals)
    for index in range(1, len(ordered)):  # noqa: SIM110
        # 2. Current start before previous end → conflict.
        if ordered[index][0] < ordered[index - 1][1]:
            return False
    return True


print(can_attend([[0, 30], [5, 10], [15, 20]]))
print(can_attend([[7, 10], [2, 4]]))
print(can_attend([[1, 2], [2, 3]]))
print(can_attend([]))


False
True
True
True


## Trace

`[[0, 30], [5, 10], [15, 20]]` sorted is the same order:

| index | interval | previous end | `start < prev_end`? |
|---|---|---|---|
| 1 | `[5, 10]` | 30 | 5 < 30 → **False** (cannot attend) |

`[[7, 10], [2, 4]]` sorts to `[[2, 4], [7, 10]]`; `7 < 4` is false, so **True**.


## Cost

Sorting dominates: time grows like `n log n`. The scan is linear. Extra memory is the sorted copy. Pairwise checks pay about `n²` overlap tests.

If intervals are already sorted by start, drop the sort and scan in linear time. Challenge inputs rarely arrive sorted, so budget for `n log n`. Space can be `O(1)` extra when you sort in place and the caller allows mutation; this lesson keeps `sorted(...)` so the original order stays intact for debugging.


## Variants

Not solved here: [Merge Intervals](../../challenges/problems/07-merge-intervals.ipynb), [Insert Interval](../../challenges/problems/25-insert-interval.ipynb), [Meeting Rooms II](../../challenges/problems/68-meeting-rooms-ii.ipynb).


## Common mistake

This bug is silent. The code treats a shared endpoint as a conflict by using `<=` instead of `<`. There is no traceback. The print is `False`, and the answer is `True`.


In [3]:
def can_attend_wrong(intervals):
    ordered = sorted(intervals)
    for index in range(1, len(ordered)):  # noqa: SIM110
        if ordered[index][0] <= ordered[index - 1][1]:  # touching counted as overlap
            return False
    return True


print(can_attend_wrong([[1, 2], [2, 3]]))


False


The fix uses `<` so equal endpoints are allowed. Run `can_attend([[1, 2], [2, 3]])` and you get `True`.


In [4]:
print(can_attend([[1, 2], [2, 3]]))


True


## Python idioms

`sorted(intervals)` sorts pairs lexicographically, which is by start then end. `all(...)` can express the scan as one expression; an early `return False` short-circuits the same way and reads clearly in a lesson.


In [5]:
def can_attend_all(intervals: list[list[int]]) -> bool:
    ordered = sorted(intervals)
    return all(
        ordered[index][0] >= ordered[index - 1][1]
        for index in range(1, len(ordered))
    )


print(can_attend_all([[7, 10], [2, 4]]))


True


## Recap

- Sort by start, then compare each start to the previous end.
- Touching endpoints are allowed when the check uses `<` for conflict.
- Pairwise checks work but grow like `n²`.

Next: [practice](01-can-attend-practice.ipynb).
